# Train/test separation, imbalance, and shortcuts

**Weeks:** 17–18 · **Core question:** When does an impressive accuracy estimate mislead us?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

Good experimental design matters as much as model architecture. In biological data, related images may come from the same patient, well, slide, or instrument. Splitting correlated observations at image level can leak information across training and testing.

**Key vocabulary:** overfitting, leakage, confounding, held-out data, balanced accuracy.

**Learning goal:** Design a defensible split for biomedical images with correlated observations.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix,balanced_accuracy_score
rng=np.random.default_rng(10)
# 60 synthetic patients; 4 nearly-identical measurements per patient.
patient_label=rng.integers(0,2,60)
identity=rng.normal(size=(60,12))
X=np.repeat(identity,4,axis=0)+rng.normal(0,.02,(240,12))
y=np.repeat(patient_label,4)
patient_id=np.repeat(np.arange(60),4)

### Experiment 2 of 3

In [ ]:
a,b=train_test_split(np.arange(len(y)),test_size=.3,random_state=2,stratify=y)
model=KNeighborsClassifier(n_neighbors=1).fit(X[a],y[a]);leaky=balanced_accuracy_score(y[b],model.predict(X[b]))
pt_train,pt_test=train_test_split(np.arange(60),test_size=.3,random_state=2)
a=np.flatnonzero(np.isin(patient_id,pt_train)); b=np.flatnonzero(np.isin(patient_id,pt_test))
model=KNeighborsClassifier(n_neighbors=1).fit(X[a],y[a]);clean=balanced_accuracy_score(y[b],model.predict(X[b]))
print('Leaky image-level split:',leaky,'Patient-level split:',clean)

### Experiment 3 of 3

In [ ]:
assert set(patient_id[a]).isdisjoint(set(patient_id[b]))
print('Confusion matrix (patient-level):\n',confusion_matrix(y[b],model.predict(X[b])))

## Exercises — complete independently
1. Repeat using several random seeds. Is leakage consistently optimistic?
2. Describe what must be grouped when images come from the same patient or microscopy plate.

## Check your understanding

**Exit ticket:** Design a defensible split for biomedical images with correlated observations.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)